# ts_mean

> 对应代码：`EconometricsCode/code_in_notes/Chap.27/ts_mean.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.27`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.27")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

In [ ]:
%%stata
set seed 505
clear

我们定义一个模拟程序 `tsmean`：生成一个样本量为 `obs`（默认 300）的 AR(1) 序列 y，其自回归系数为 `beta`（默认 0.8）、截距为 `mu`（默认 0），即 $y_t=\mu+\beta y_{t-1}+\varepsilon_t$，$\varepsilon_t$ 为标准正态。`tempvar` 保证临时变量名不与现有变量冲突；先令所有观测等于 $\mu$，再从第二期起按 AR(1) 递推生成。

接下来对同一个序列做两种均值推断：先用普通 `reg y`（只含常数项）得到样本均值及其 OLS 标准误——这个标准误在误差自相关时是无效的；再用 `newey y, lag(20)` 计算 Newey–West 的 HAC 标准误，它对任意形式的自相关与异方差都稳健。两个标准误都以 rclass 标量返回，供模拟比较。

In [ ]:
%%stata
cap program drop tsmean
program define tsmean, rclass
	syntax [, obs(integer 300) beta(real 0.8) mu(real 0)]
	drop _all
	set obs `obs'
	tempvar y t
	gen `t'=_n
	tsset `t'
	gen `y'=`mu'
	replace `y'=`beta'*L.`y'+rnormal() if `t'>1
	quietly: reg `y'
	return scalar sample_mean=_b[_cons]
	return scalar unadjusted_se=_se[_cons]
	quietly: newey `y', lag(20)
	return scalar adjusted_se=_se[_cons]
end

用 `simulate` 重复 1000 次上述实验，收集每次的样本均值、未调整标准误与 Newey–West 标准误，最后 `su` 汇总。我们将看到：在 AR(1) 序列正自相关下，未调整的 OLS 标准误系统性偏小，而 HAC 标准误更大、更接近真实的抽样变异。

In [ ]:
%%stata
simulate m=r(sample_mean) se=r(unadjusted_se) ase=r(adjusted_se), reps(1000): tsmean
su